In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import re


text = """machine learning models learn patterns from data.
sequence models process data step by step.
recurrent neural networks are designed for sequential tasks.
rnn models maintain hidden states across time steps.

long short term memory networks solve long dependency problems.
lstm uses gates to control information flow.
gru models simplify the lstm architecture.
sequence prediction is useful in many applications.

language modeling predicts the next word in a sentence.
speech recognition processes audio sequences.
time series forecasting predicts future values.
music generation creates new melodies.

generative models learn probability distributions.
they generate new samples similar to training data.
sequence generation is widely used in artificial intelligence.
deep learning improves sequence modeling performance.
"""

text = re.sub(r'[^\w\s]', '', text)

tokens = text.lower().split()


vocab = sorted(set(tokens))
word2idx = {w:i for i,w in enumerate(vocab)}
idx2word = {i:w for w,i in word2idx.items()}


seq_length = 4
X, y = [], []

for i in range(len(tokens) - seq_length):
    X.append([word2idx[w] for w in tokens[i:i+seq_length]])
    y.append(word2idx[tokens[i+seq_length]])

X = torch.tensor(X)
y = torch.tensor(y)


class LSTMModel(nn.Module):
    def __init__(self, vocab_size, embed_size=32, hidden_size=64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.lstm = nn.LSTM(embed_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.lstm(x)
        out = self.fc(out[:, -1, :])
        return out

lstm_model = LSTMModel(len(vocab))

# TRAIN LSTM
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(lstm_model.parameters(), lr=0.01)

for epoch in range(200):
    optimizer.zero_grad()
    output = lstm_model(X)
    loss = criterion(output, y)
    loss.backward()
    optimizer.step()

    if epoch % 50 == 0:
        print(f"LSTM Epoch {epoch}, Loss: {loss.item()}")

# GENERATE (LSTM)
def generate_lstm(seed, length=10):
    lstm_model.eval()
    words = seed.split()

    for _ in range(length):
        inp = torch.tensor([[word2idx[w] for w in words[-seq_length:]]])
        with torch.no_grad():
            out = lstm_model(inp)

        prob = torch.softmax(out, dim=-1)
        next_word = idx2word[torch.multinomial(prob, 1).item()]
        words.append(next_word)

    return " ".join(words)


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2) * (-np.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)

        self.pe = pe.unsqueeze(0)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class TransformerModel(nn.Module):
    def __init__(self, vocab_size, d_model=64, nhead=4, num_layers=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos = PositionalEncoding(d_model)

        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.fc = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        x = self.pos(x)
        x = x.permute(1, 0, 2)  # (seq_len, batch, d_model)

        out = self.transformer(x)
        out = self.fc(out[-1])  # last token

        return out

transformer_model = TransformerModel(len(vocab))

# TRAIN TRANSFORMER
optimizer = optim.Adam(transformer_model.parameters(), lr=0.01)

for epoch in range(200):
    optimizer.zero_grad()
    output = transformer_model(X)
    loss = criterion(output, y)
    loss.backward()
    optimizer.step()

    if epoch % 50 == 0:
        print(f"Transformer Epoch {epoch}, Loss: {loss.item()}")

# GENERATE (Transformer)
def generate_transformer(seed, length=10):
    transformer_model.eval()
    words = seed.split()

    for _ in range(length):
        inp = torch.tensor([[word2idx[w] for w in words[-seq_length:]]])
        with torch.no_grad():
            out = transformer_model(inp)

        prob = torch.softmax(out, dim=-1)
        next_word = idx2word[torch.multinomial(prob, 1).item()]
        words.append(next_word)

    return " ".join(words)


print("\n🔹 LSTM Generated:")
print(generate_lstm("machine learning models learn"))

print("\n🔹 Transformer Generated:")
print(generate_transformer("deep learning improves sequence"))

LSTM Epoch 0, Loss: 4.4524078369140625
LSTM Epoch 50, Loss: 0.0036298807244747877
LSTM Epoch 100, Loss: 0.001503323088400066
LSTM Epoch 150, Loss: 0.00102152896579355


/tmp/ipykernel_10828/2679378149.py:119: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)


Transformer Epoch 0, Loss: 4.642543315887451
Transformer Epoch 50, Loss: 0.0024337044451385736
Transformer Epoch 100, Loss: 0.0010071127908304334
Transformer Epoch 150, Loss: 0.0006951519171707332

🔹 LSTM Generated:
machine learning models learn patterns from data sequence models process data step by step

🔹 Transformer Generated:
deep learning improves sequence modeling performance performance audio sequences time series forecasting predicts future
